In [ ]:
# Ekstrak Landmark untuk Kata
import cv2
import mediapipe as mp
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

# ==========================================================
# KONFIGURASI
# ==========================================================
DATASET_PATH = "dataset_kata_video"
OUTPUT_PATH  = "dataset_npy"
TEMP_PATH    = os.path.join(OUTPUT_PATH, "temp_videos") # Folder simpan progress per video

SEQ_LENGTH = 30   # jumlah frame per video

LABELS = [
    "berasal",
    "berpikir",
    "maaf",
    "makan",
    "mandi",
    "nama",
    "perkenalkan",
    "saya",
    "tidur",
    "tolong",
    "tunggu",
    "umur",
    "berteman",
    "sementara",
    "santai aja",
    "bareng",
]

FACE_POINTS = [1, 33, 61, 199, 263]
POSE_POINTS = [11, 12, 13, 14, 15]

# ==========================================================
# MEDIAPIPE SETUP
# ==========================================================
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

# ==========================================================
# EKSTRAK FITUR 1 FRAME
# ==========================================================
def extract_features(result):
    data = []
    if result.left_hand_landmarks:
        for lm in result.left_hand_landmarks.landmark: data += [lm.x, lm.y]
    else: data += [0.0] * 42

    if result.right_hand_landmarks:
        for lm in result.right_hand_landmarks.landmark: data += [lm.x, lm.y]
    else: data += [0.0] * 42

    if result.face_landmarks:
        for i in FACE_POINTS:
            lm = result.face_landmarks.landmark[i]
            data += [lm.x, lm.y]
    else: data += [0.0] * 10

    if result.pose_landmarks:
        for i in POSE_POINTS:
            lm = result.pose_landmarks.landmark[i]
            data += [lm.x, lm.y]
    else: data += [0.0] * 10

    return data

# ==========================================================
# PROSES SATU VIDEO -> SEQUENCE LANDMARK
# ==========================================================
def process_video(video_path, seq_length=SEQ_LENGTH):
    cap = cv2.VideoCapture(video_path)
    sequence = []

    while True:
        ret, frame = cap.read()
        if not ret: break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = holistic.process(rgb)
        feat = extract_features(result)
        sequence.append(feat)

    cap.release()
    sequence = np.array(sequence)

    if len(sequence) == 0:
        return np.zeros((seq_length, 104))

    if len(sequence) < seq_length:
        pad = np.zeros((seq_length - len(sequence), 104))
        sequence = np.vstack([sequence, pad])
    else:
        sequence = sequence[:seq_length]

    return sequence

# ==========================================================
# LOOP UTAMA (RESUME LEVEL VIDEO)
# ==========================================================
def main():
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    os.makedirs(TEMP_PATH, exist_ok=True)

    for label_idx, label in enumerate(LABELS):
        label_path = os.path.join(DATASET_PATH, label)

        if not os.path.exists(label_path):
            print(f"[WARN] Folder tidak ditemukan: {label_path}, dilewati.")
            continue

        videos = [
            f for f in os.listdir(label_path)
            if f.lower().endswith((".mp4", ".avi", ".mov"))
        ]

        if len(videos) == 0: continue

        print(f"Mengecek label '{label}' ({len(videos)} video)...")

        for vname in videos:
            vpath = os.path.join(label_path, vname)
            
            # Membuat nama file unik per video, misal: berasal_bantu_001.mp4.npy
            video_base_name = os.path.splitext(vname)[0]
            temp_file_name = f"{label}_{video_base_name}.npy"
            temp_file_path = os.path.join(TEMP_PATH, temp_file_name)

            # Cek apakah video spesifik ini sudah pernah diekstrak
            if os.path.exists(temp_file_path):
                continue  # SKIP video ini, lanjut ke video berikutnya di folder yang sama

            print(f"  -> Mengekstrak: {vname}")
            seq = process_video(vpath)

            # Simpan fitur video ini beserta index labelnya
            np.save(temp_file_path, {"features": seq, "label_idx": label_idx})

    # ==========================================================
    # PENGGABUNGAN SEMUA FILE VIDEO MENJADI X.NPY & Y.NPY (FIXED)
    # ==========================================================
    print("\n[INFO] Menggabungkan seluruh file video menjadi dataset final...")
    X_all = []
    y_all = []

    # Iterasi ulang sesuai urutan LABELS agar susunan datanya rapi
    for label_idx, label in enumerate(LABELS):
        label_path = os.path.join(DATASET_PATH, label)
        if not os.path.exists(label_path): continue
        
        videos = [f for f in os.listdir(label_path) if f.lower().endswith((".mp4", ".avi", ".mov"))]
        
        for vname in videos:
            video_base_name = os.path.splitext(vname)[0]
            temp_file_name = f"{label}_{video_base_name}.npy"
            temp_file_path = os.path.join(TEMP_PATH, temp_file_name)

            if os.path.exists(temp_file_path):
                try:
                    # Load data dengan pengaman allow_pickle
                    data = np.load(temp_file_path, allow_pickle=True).item()
                    X_all.append(data["features"])
                    y_all.append(data["label_idx"])
                except Exception as e:
                    # JIKA FILE CORRUPT / 0 KB -> HAPUS OTOMATIS
                    print(f"[WARN] File corrupt terdeteksi: '{temp_file_name}'. Menghapus file...")
                    try:
                        os.remove(temp_file_path)
                    except:
                        pass
                    print(f"       -> Sistem akan mengekstrak ulang video ini pada pengerjaan berikutnya: {vname}")

    if len(X_all) > 0:
        X = np.array(X_all)
        y = np.array(y_all)

        print(f"\nTotal data final: {X.shape}")
        print(f"Total label final: {y.shape}")

        np.save(os.path.join(OUTPUT_PATH, "X.npy"), X)
        np.save(os.path.join(OUTPUT_PATH, "y.npy"), y)

        with open(os.path.join(OUTPUT_PATH, "label_map.txt"), "w") as f:
            for idx, label in enumerate(LABELS):
                f.write(f"{idx}: {label}\n")

        print(f"\n[OK] Selesai! Data final diperbarui di '{OUTPUT_PATH}/'")
    else:
        print("[ERROR] Tidak ada data video yang berhasil diekstrak.")

if __name__ == "__main__":
    main()